# Notebook 05 — RT-DETR Training

**Dataset:** B (Static/Watchtower) | **Paradigm:** Real-Time Transformer  
**Model:** `PekingU/rtdetr-r50vd` via HuggingFace Transformers

## Config
- Image: 640×640 | Batch: 4 | Epochs: 50 | Optimizer: AdamW lr=0.0001 | FP16: ✅

## Why RT-DETR for Dataset B
Full cross-attention (not windowed) captures global scene context — critical for distinguishing
distant smoke columns from cloud formations in panoramic watchtower imagery.

In [ ]:
# ── Install (Colab) ───────────────────────────────────────────────────────
# !pip install transformers>=4.40.0 pycocotools --quiet

In [ ]:
import torch
from pathlib import Path
from transformers import RTDetrForObjectDetection, RTDetrImageProcessor
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from tqdm import tqdm
import json
from PIL import Image

ROOT        = Path('..')
DATA_B      = ROOT / 'data' / 'dataset-b'
RESULTS_DIR = ROOT / 'results' / 'experiment-b' / 'rtdetr'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE    = 'cuda' if torch.cuda.is_available() else 'cpu'
NUM_CLASSES = 2  # fire, smoke
print(f"Device: {DEVICE}")

In [ ]:
# ── Load model and processor ───────────────────────────────────────────────
MODEL_NAME = "PekingU/rtdetr-r50vd"
processor  = RTDetrImageProcessor.from_pretrained(MODEL_NAME)
model      = RTDetrForObjectDetection.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_CLASSES,
    ignore_mismatched_sizes=True  # re-initialise classification head
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model: {MODEL_NAME}")
print(f"Parameters: {total_params/1e6:.1f}M")

In [ ]:
# ── Dataset class ─────────────────────────────────────────────────────────
class FireDatasetCOCO(Dataset):
    def __init__(self, img_dir, ann_json, processor, size=(640, 640)):
        self.img_dir   = Path(img_dir)
        self.processor = processor
        self.size      = size
        with open(ann_json) as f:
            coco = json.load(f)
        self.images  = {img['id']: img for img in coco['images']}
        self.img_ids = [img['id'] for img in coco['images']]
        self.anns    = {}
        for ann in coco['annotations']:
            self.anns.setdefault(ann['image_id'], []).append(ann)

    def __len__(self):
        return len(self.img_ids)

    def __getitem__(self, idx):
        img_id  = self.img_ids[idx]
        img_inf = self.images[img_id]
        image   = Image.open(self.img_dir / img_inf['file_name']).convert('RGB')
        image   = image.resize(self.size)
        anns    = self.anns.get(img_id, [])
        boxes   = [[a['bbox'][0], a['bbox'][1],
                    a['bbox'][0]+a['bbox'][2],
                    a['bbox'][1]+a['bbox'][3]] for a in anns]
        labels  = [a['category_id'] for a in anns]
        target  = {'boxes': boxes, 'class_labels': labels, 'image_id': img_id}
        inputs  = self.processor(images=image, annotations=target, return_tensors='pt')
        return {k: v.squeeze(0) for k, v in inputs.items()}

In [ ]:
# ── Training loop ─────────────────────────────────────────────────────────
def collate_fn(batch):
    return {k: [b[k] for b in batch] for k in batch[0]}

train_ds = FireDatasetCOCO(
    DATA_B / 'images' / 'train' / 'images',
    DATA_B / 'annotations' / 'train.json',
    processor, size=(640, 640)
)
val_ds = FireDatasetCOCO(
    DATA_B / 'images' / 'val' / 'images',
    DATA_B / 'annotations' / 'val.json',
    processor, size=(640, 640)
)
train_dl = DataLoader(train_ds, batch_size=4, shuffle=True, collate_fn=collate_fn, num_workers=2)
val_dl   = DataLoader(val_ds,   batch_size=4, shuffle=False, collate_fn=collate_fn, num_workers=2)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scaler    = GradScaler()
EPOCHS    = 50

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch in tqdm(train_dl, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        pixel_values = torch.stack(batch['pixel_values']).to(DEVICE)
        labels = [{k: v.to(DEVICE) for k, v in t.items()} for t in batch['labels']]
        optimizer.zero_grad()
        with autocast():
            outputs = model(pixel_values=pixel_values, labels=labels)
            loss = outputs.loss
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
    print(f"Epoch {epoch+1} — Loss: {total_loss/len(train_dl):.4f}")
    # Save checkpoint every 10 epochs
    if (epoch+1) % 10 == 0:
        model.save_pretrained(RESULTS_DIR / f'checkpoint_ep{epoch+1}')

model.save_pretrained(RESULTS_DIR / 'best')
print("✅ RT-DETR training complete")